In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Étape 1 : Définir les modèles Pydantic pour les sorties structurées

Ces modèles définissent le **schéma** que les agents retourneront. Utiliser `response_format` avec Pydantic assure :
- ✅ Extraction de données sûre par type
- ✅ Validation automatique
- ✅ Pas d'erreurs d'analyse des réponses en texte libre
- ✅ Routage conditionnel facile basé sur les champs


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Étape 2 : Créer l’outil de réservation d’hôtel

Cet outil est ce que l’**availability_agent** appellera pour vérifier si des chambres sont disponibles. Nous utilisons le décorateur `@ai_function` pour :
- Convertir une fonction Python en un outil accessible par IA
- Générer automatiquement un schéma JSON pour le LLM
- Gérer la validation des paramètres
- Permettre l’invocation automatique par les agents

Pour cette démo :
- **Stockholm, Seattle, Tokyo, Londres, Amsterdam** → Ont des chambres ✅
- **Toutes les autres villes** → Pas de chambres ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Étape 3 : Définir les fonctions de condition pour le routage

Ces fonctions inspectent la réponse de l'agent et déterminent quel chemin emprunter dans le flux de travail.

**Motif clé :**
1. Vérifier si le message est `AgentExecutorResponse`
2. Analyser la sortie structurée (modèle Pydantic)
3. Retourner `True` ou `False` pour contrôler le routage

Le flux de travail évaluera ces conditions sur les **arêtes** pour décider quel exécuteur invoquer ensuite.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Étape 4 : Créer un exécuteur d'affichage personnalisé

Les exécuteurs sont des composants de workflow qui effectuent des transformations ou des effets secondaires. Nous utilisons le décorateur `@executor` pour créer un exécuteur personnalisé qui affiche le résultat final.

**Concepts clés :**
- `@executor(id="...")` - Enregistre une fonction en tant qu'exécuteur de workflow
- `WorkflowContext[Never, str]` - Indications de type pour l'entrée/sortie
- `ctx.yield_output(...)` - Produit le résultat final du workflow


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Étape 5 : Charger les variables d'environnement

Configurez le client LLM. Cet exemple fonctionne avec :
- **Modèles GitHub** (offre gratuite avec jeton GitHub)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Étape 6 : Créer des agents IA avec des sorties structurées

Nous créons **trois agents spécialisés**, chacun enveloppé dans un `AgentExecutor` :

1. **availability_agent** - Vérifie la disponibilité des hôtels en utilisant l’outil
2. **alternative_agent** - Suggère des villes alternatives (lorsqu’il n’y a pas de chambres)
3. **booking_agent** - Encourage la réservation (lorsque des chambres sont disponibles)

**Caractéristiques principales :**
- `tools=[hotel_booking]` - Fournit l’outil à l’agent
- `response_format=PydanticModel` - Force une sortie JSON structurée
- `AgentExecutor(..., id="...")` - Enveloppe l’agent pour l’utiliser dans le flux de travail


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Étape 7 : Construire le flux de travail avec des arêtes conditionnelles

Nous utilisons maintenant `WorkflowBuilder` pour construire le graphe avec un routage conditionnel :

**Structure du flux de travail :**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Méthodes clés :**
- `.set_start_executor(...)` - Définit le point d'entrée
- `.add_edge(from, to, condition=...)` - Ajoute une arête conditionnelle
- `.build()` - Finalise le flux de travail


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Étape 8 : Exécuter le cas de test 1 - Ville SANS disponibilité (Paris)

Testons le chemin **sans disponibilité** en demandant des hôtels à Paris (qui n'a pas de chambres dans notre simulation).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Étape 9 : Exécuter le cas de test 2 - Ville AVEC disponibilité (Stockholm)

Testons maintenant le chemin **disponibilité** en demandant des hôtels à Stockholm (qui dispose de chambres dans notre simulation).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Points clés et étapes suivantes

### ✅ Ce que vous avez appris :

1. **Modèle WorkflowBuilder**
   - Utilisez `.set_start_executor()` pour définir le point d'entrée
   - Utilisez `.add_edge(from, to, condition=...)` pour le routage conditionnel
   - Appelez `.build()` pour finaliser le workflow

2. **Routage Conditionnel**
   - Les fonctions condition inspectent `AgentExecutorResponse`
   - Analysez les sorties structurées pour prendre des décisions de routage
   - Retournez `True` pour activer une liaison, `False` pour la passer

3. **Intégration d'Outils**
   - Utilisez `@ai_function` pour convertir des fonctions Python en outils IA
   - Les agents appellent les outils automatiquement lorsque nécessaire
   - Les outils renvoient du JSON que les agents peuvent analyser

4. **Sorties Structurées**
   - Utilisez des modèles Pydantic pour une extraction de données typée sécurisée
   - Définissez `response_format=MyModel` lors de la création d'agents
   - Analysez les réponses avec `Model.model_validate_json()`

5. **Exécuteurs Personnalisés**
   - Utilisez `@executor(id="...")` pour créer des composants de workflow
   - Les exécuteurs peuvent transformer des données ou effectuer des effets secondaires
   - Utilisez `ctx.yield_output()` pour produire des résultats du workflow

### 🚀 Applications dans le Monde Réel :

- **Réservation de voyages** : Vérifier la disponibilité, suggérer des alternatives, comparer les options
- **Service Client** : Router selon le type de problème, le sentiment, la priorité
- **E-commerce** : Vérifier les stocks, suggérer des alternatives, traiter les commandes
- **Modération de contenu** : Router selon les scores de toxicité, les signalements utilisateurs
- **Workflows d'approbation** : Router selon le montant, le rôle utilisateur, le niveau de risque
- **Traitement multi-étapes** : Router selon la qualité des données, la complétude

### 📚 Étapes suivantes :

- Ajouter des conditions plus complexes (critères multiples)
- Mettre en œuvre des boucles avec gestion d'état du workflow
- Ajouter des sous-workflows pour composants réutilisables
- Intégrer avec des API réelles (réservation d'hôtels, systèmes de gestion de stock)
- Ajouter la gestion des erreurs et des chemins de secours
- Visualiser les workflows avec les outils de visualisation intégrés


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
